# SpaceX Falcon 9 First Stage Landing Prediction
## 3. Visualization & Interactive Mapping with Folium

This notebook covers:
- Interactive Folium map visualization
- Launch site marking and clustering
- Success/failure visualization on map
- Distance calculations

## Import Libraries

In [1]:
!pip install folium

In [2]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster, MousePosition, HeatMap
from folium.features import DivIcon
import warnings
warnings.filterwarnings('ignore')

# Haversine distance calculation
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometers
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

print('✅ Libraries imported')

✅ Libraries imported


## Load Data

In [3]:
# Create sample dataset for demonstration
launch_sites = pd.DataFrame({
    'LaunchSite': [
        'CCAFS LC-40',
        'KSC LC-39A',
        'VAFB SLC-4E',
        'CCAFS SLC-40'
    ],
    'Latitude': [28.5621, 28.6329, 34.6329, 28.5621],
    'Longitude': [-80.5774, -80.6270, -120.6102, -80.5774],
    'Flights': [50, 15, 12, 5]
})

# Create launch records
launches = pd.DataFrame({
    'LaunchSite': ['CCAFS LC-40']*30 + ['KSC LC-39A']*12 + ['VAFB SLC-4E']*10 + ['CCAFS SLC-40']*5,
    'Latitude': [28.5621]*30 + [28.6329]*12 + [34.6329]*10 + [28.5621]*5,
    'Longitude': [-80.5774]*30 + [-80.6270]*12 + [-120.6102]*10 + [-80.5774]*5,
    'Class': np.random.choice([0, 1], 57, p=[0.3, 0.7])
})

print(f'✅ Dataset loaded: {len(launches)} launches')
print(f'Launch sites: {launches["LaunchSite"].nunique()}')

✅ Dataset loaded: 57 launches
Launch sites: 4


## Task 1: Basic Launch Sites Map

In [4]:
# NASA Johnson Space Center coordinates
nasa_lat = 29.5497
nasa_lon = -95.0865

# Create base map
site_map = folium.Map(
    location=[nasa_lat, nasa_lon],
    zoom_start=5,
    tiles='OpenStreetMap'
)

# Add NASA marker
folium.Marker(
    [nasa_lat, nasa_lon],
    popup='NASA Johnson Space Center',
    tooltip='NASA JSC',
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(site_map)

# Add launch sites
for idx, row in launch_sites.iterrows():
    folium.Marker(
        [row['Latitude'], row['Longitude']],
        popup=f"{row['LaunchSite']}<br>Flights: {row['Flights']}",
        tooltip=row['LaunchSite'],
        icon=folium.Icon(color='blue', icon='rocket')
    ).add_to(site_map)

# Add mouse position
MousePosition().add_to(site_map)

# Save map
site_map.save('launch_sites_map.html')
print('✅ Saved: launch_sites_map.html')
site_map

✅ Saved: launch_sites_map.html


## Task 2: Success/Failure Visualization Map

In [5]:
# Create success/failure map
success_map = folium.Map(
    location=[nasa_lat, nasa_lon],
    zoom_start=5,
    tiles='OpenStreetMap'
)

# Add cluster
marker_cluster = MarkerCluster().add_to(success_map)

# Add launches with color coding
for idx, row in launches.iterrows():
    # Color based on outcome
    color = 'green' if row['Class'] == 1 else 'red'
    outcome = 'Success' if row['Class'] == 1 else 'Failed'
    
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']],
        radius=6,
        popup=f"{row['LaunchSite']}<br>Outcome: {outcome}",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=2
    ).add_to(success_map)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 180px; height: 100px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p style="margin: 0;"><b>Landing Outcome</b></p>
<p style="margin: 5px; color: green;">● Success</p>
<p style="margin: 5px; color: red;">● Failed</p>
</div>
'''
success_map.get_root().html.add_child(folium.Element(legend_html))

# Save map
success_map.save('success_failure_map.html')
print('✅ Saved: success_failure_map.html')
success_map

✅ Saved: success_failure_map.html


## Task 3: Distance Calculations

In [6]:
# Key locations
locations = {
    'NASA JSC': (29.5497, -95.0865),
    'KSC': (28.5293, -80.6500),
    'Equator': (0, 0),
    'North Pole': (90, 0)
}

# Calculate distances from each launch site
print('\nDistance Calculations from Launch Sites:')
print('='*70)

for site_idx, site_row in launch_sites.iterrows():
    print(f'\n{site_row["LaunchSite"]}')
    print('-'*50)
    
    for loc_name, (loc_lat, loc_lon) in locations.items():
        dist = haversine(site_row['Latitude'], site_row['Longitude'], loc_lat, loc_lon)
        print(f'  Distance to {loc_name:15s}: {dist:8.2f} km')


Distance Calculations from Launch Sites:

CCAFS LC-40
--------------------------------------------------
  Distance to NASA JSC       :  1413.64 km
  Distance to KSC            :     7.97 km
  Distance to Equator        :  9088.27 km
  Distance to North Pole     :  6831.58 km

KSC LC-39A
--------------------------------------------------
  Distance to NASA JSC       :  1407.77 km
  Distance to KSC            :    11.74 km
  Distance to Equator        :  9093.71 km
  Distance to North Pole     :  6823.71 km

VAFB SLC-4E
--------------------------------------------------
  Distance to NASA JSC       :  2462.77 km
  Distance to KSC            :  3820.43 km
  Distance to Equator        : 12761.80 km
  Distance to North Pole     :  6156.54 km

CCAFS SLC-40
--------------------------------------------------
  Distance to NASA JSC       :  1413.64 km
  Distance to KSC            :     7.97 km
  Distance to Equator        :  9088.27 km
  Distance to North Pole     :  6831.58 km


## Task 4: Heatmap Visualization

In [7]:
# Create heatmap showing launch density
heatmap = folium.Map(
    location=[nasa_lat, nasa_lon],
    zoom_start=5,
    tiles='OpenStreetMap'
)

# Prepare heatmap data
heat_data = [[row['Latitude'], row['Longitude']] for idx, row in launches.iterrows()]

# Add heatmap layer
HeatMap(heat_data, radius=30, blur=15, max_zoom=1).add_to(heatmap)

# Save heatmap
heatmap.save('launch_density_heatmap.html')
print('✅ Saved: launch_density_heatmap.html')
heatmap

✅ Saved: launch_density_heatmap.html


## Summary Statistics

In [8]:
print('\n' + '='*70)
print('MAPPING ANALYSIS SUMMARY')
print('='*70)

print(f'\nLaunch Sites: {len(launch_sites)}')
for idx, row in launch_sites.iterrows():
    print(f'  - {row["LaunchSite"]:20s} ({row["Latitude"]:7.2f}, {row["Longitude"]:8.2f})')

print(f'\nTotal Launches Tracked: {len(launches)}')
print(f'Successful: {(launches["Class"] == 1).sum()}')
print(f'Failed: {(launches["Class"] == 0).sum()}')
print(f'Success Rate: {launches["Class"].mean():.2%}')

print(f'\nMaps Generated:')
print(f'  ✅ launch_sites_map.html')
print(f'  ✅ success_failure_map.html')
print(f'  ✅ launch_density_heatmap.html')
print(f'\n✅ Mapping Analysis Complete')


MAPPING ANALYSIS SUMMARY

Launch Sites: 4
  - CCAFS LC-40          (  28.56,   -80.58)
  - KSC LC-39A           (  28.63,   -80.63)
  - VAFB SLC-4E          (  34.63,  -120.61)
  - CCAFS SLC-40         (  28.56,   -80.58)

Total Launches Tracked: 57
Successful: 41
Failed: 16
Success Rate: 71.93%

Maps Generated:
  ✅ launch_sites_map.html
  ✅ success_failure_map.html
  ✅ launch_density_heatmap.html

✅ Mapping Analysis Complete
